# Azure OpenAI Response API — Stateless Cross-Instance Test

This notebook tests whether the **Response API state** (i.e. `response_id`) is portable across
different Azure OpenAI (Foundry) instances deployed in **Sweden Central**.

## Test Plan

| # | Test | Description |
|---|------|-------------|
| 1 | **Baseline** | Send a request to Instance 1, retrieve the response using the same instance. |
| 1b | **Cross-Model Retrieve** | Create with gpt-4.1-mini on Instance 1, retrieve using gpt-5-mini on Instance 1. |
| 1c | **Cross-Model Chaining** | Create with gpt-4.1-mini on Instance 1, chain with gpt-5-mini on Instance 1. |
| 2 | **Cross-Instance Retrieve** | Send a request to Instance 1, retrieve the response from Instance 2 & 3. |
| 3 | **Cross-Instance Chaining** | Send a request to Instance 1, chain with `previous_response_id` on Instance 2 & 3. |
| 4 | **Background Mode Baseline** | Send a `background=True` request to Instance 1, poll until complete. |
| 4b | **Background Retrieve (same)** | Retrieve the background response from Instance 1 (same instance). |
| 4c | **Background Cross-Model Retrieve** | Retrieve the background response (gpt-4.1-mini) using gpt-5-mini on Instance 1. |
| 4d | **Background Cross-Model Chaining** | Chain with gpt-5-mini using background `previous_response_id` from gpt-4.1-mini on Instance 1. |
| 5 | **Background Cross-Instance Retrieve** | Send `background=True` to Instance 1, retrieve from Instance 2 & 3. |
| 6 | **Background Cross-Instance Chaining** | Send `background=True` to Instance 1, chain on Instance 2 & 3. |

Each test logs the outcome (success / failure + error details).

## 0. Setup & Configuration

In [43]:
# Install / upgrade dependencies
%pip install --upgrade openai python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


In [44]:
import os
import json
import time
from datetime import datetime
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Validate that all required env vars are set
required_vars = [
    "AZURE_OPENAI_ENDPOINT_1", "AZURE_OPENAI_API_KEY_1",
    "AZURE_OPENAI_ENDPOINT_2", "AZURE_OPENAI_API_KEY_2",
    "AZURE_OPENAI_ENDPOINT_3", "AZURE_OPENAI_API_KEY_3",
    "AZURE_OPENAI_DEPLOYMENT_NAME",
    "AZURE_OPENAI_DEPLOYMENT_NAME_2",
]
missing = [v for v in required_vars if not os.getenv(v)]
if missing:
    raise EnvironmentError(f"Missing environment variables: {', '.join(missing)}")

DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
DEPLOYMENT_2 = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME_2")

print(f"Deployment 1    : {DEPLOYMENT}")
print(f"Deployment 2    : {DEPLOYMENT_2}")
print(f"Instance 1      : {os.getenv('AZURE_OPENAI_ENDPOINT_1')}")
print(f"Instance 2      : {os.getenv('AZURE_OPENAI_ENDPOINT_2')}")
print(f"Instance 3      : {os.getenv('AZURE_OPENAI_ENDPOINT_3')}")

Deployment 1    : gpt-4-1-mini
Deployment 2    : gpt-5-mini
Instance 1      : https://ais-fnd1-6ohufxbigpdyo.openai.azure.com/
Instance 2      : https://ais-fnd2-6ohufxbigpdyo.openai.azure.com/
Instance 3      : https://ais-fnd3-6ohufxbigpdyo.openai.azure.com/


In [45]:
def make_client(instance_num: int) -> OpenAI:
    """Create an OpenAI client for the given Foundry instance (1, 2, or 3)."""
    endpoint = os.getenv(f"AZURE_OPENAI_ENDPOINT_{instance_num}").rstrip("/")
    api_key = os.getenv(f"AZURE_OPENAI_API_KEY_{instance_num}")
    return OpenAI(
        api_key=api_key,
        base_url=f"{endpoint}/openai/v1/",
        default_headers={"api-key": api_key},
    )

client1 = make_client(1)
client2 = make_client(2)
client3 = make_client(3)

clients = {1: client1, 2: client2, 3: client3}

print("All three OpenAI clients created successfully.")

All three OpenAI clients created successfully.


In [46]:
# ── Helper utilities ──────────────────────────────────────────────────

results = []  # collects test outcomes


def print_response_json(response, label: str = "Response"):
    """Pretty-print a Response API object as formatted JSON."""
    print(f"\n📋 {label} (JSON):")
    print(json.dumps(response.model_dump(), indent=2, default=str))
    print()


def log_result(test_name: str, success: bool, details: str = ""):
    """Append a test result and print it."""
    status = "PASS" if success else "FAIL"
    entry = {
        "test": test_name,
        "status": status,
        "details": details,
        "timestamp": datetime.utcnow().isoformat(),
    }
    results.append(entry)
    colour = "\033[92m" if success else "\033[91m"
    print(f"{colour}[{status}]\033[0m {test_name}")
    if details:
        print(f"       ↳ {details}")


def wait_for_background(client: OpenAI, response_id: str, timeout: int = 120) -> object:
    """Poll a background response until it reaches a terminal state."""
    start = time.time()
    resp = client.responses.retrieve(response_id)
    while resp.status in ("queued", "in_progress"):
        if time.time() - start > timeout:
            raise TimeoutError(f"Background response {response_id} did not complete within {timeout}s")
        time.sleep(2)
        resp = client.responses.retrieve(response_id)
    return resp

---
## 1. Baseline — Create & Retrieve on Same Instance

In [47]:
# Test 1: Create a response on Instance 1 and retrieve it from the same instance
print("Creating response on Instance 1...")
response1 = client1.responses.create(
    model=DEPLOYMENT,
    input="What is the capital of France? Answer in one word.",
)

response1_id = response1.id
print(f"Response ID  : {response1_id}")
print(f"Status       : {response1.status}")
print(f"Output       : {response1.output_text}")
print_response_json(response1, "Create Response (Instance 1)")

# Retrieve the same response from Instance 1
print("Retrieving response from Instance 1 (same instance)...")
try:
    retrieved = client1.responses.retrieve(response1_id)
    print_response_json(retrieved, "Retrieved Response (Instance 1)")
    log_result(
        "1. Baseline — create & retrieve on Instance 1",
        success=True,
        details=f"Retrieved output: {retrieved.output_text}",
    )
except Exception as e:
    log_result("1. Baseline — create & retrieve on Instance 1", success=False, details=str(e))

Creating response on Instance 1...
Response ID  : resp_0da64bbc724270c50069cbbb69401481969b882f95011a7888
Status       : completed
Output       : Paris

📋 Create Response (Instance 1) (JSON):
{
  "id": "resp_0da64bbc724270c50069cbbb69401481969b882f95011a7888",
  "created_at": 1774959465.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-4-1-mini",
  "object": "response",
  "output": [
    {
      "id": "msg_0da64bbc724270c50069cbbb69a3ac81968653570569fe78bf",
      "content": [
        {
          "annotations": [],
          "text": "Paris",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": null
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 1.0,
  "background": false,
  "completed_at": 1774959465.0,
  "conversation": null,
  "max_output_tok

---
## 1b. Cross-Model Retrieve — Fetch response created with gpt-4.1-mini using gpt-5-mini (same instance)

In [48]:
# Test 1b: Retrieve the response (created with gpt-4.1-mini) using gpt-5-mini on the same Instance 1
print(f"Retrieving response {response1_id} from Instance 1 using {DEPLOYMENT_2}...")
print(f"  (Response was created with model={DEPLOYMENT})")
try:
    retrieved_cross_model = client1.responses.retrieve(response1_id)
    print_response_json(retrieved_cross_model, f"Cross-Model Retrieved ({DEPLOYMENT} → {DEPLOYMENT_2}, Instance 1)")
    log_result(
        f"1b. Cross-Model Retrieve — {DEPLOYMENT} → {DEPLOYMENT_2} (Instance 1)",
        success=True,
        details=f"Retrieved output: {retrieved_cross_model.output_text}",
    )
except Exception as e:
    log_result(
        f"1b. Cross-Model Retrieve — {DEPLOYMENT} → {DEPLOYMENT_2} (Instance 1)",
        success=False,
        details=str(e),
    )

# Test 1c: Chain on the same Instance 1 using gpt-5-mini with previous_response_id from gpt-4.1-mini
print(f"\nChaining on Instance 1 with {DEPLOYMENT_2} using previous_response_id={response1_id}...")
print(f"  (Original response was created with model={DEPLOYMENT})")
try:
    chained_cross_model = client1.responses.create(
        model=DEPLOYMENT_2,
        previous_response_id=response1_id,
        input=[{"role": "user", "content": "What is the population of that city? Answer briefly."}],
    )
    print_response_json(chained_cross_model, f"Cross-Model Chained ({DEPLOYMENT} → {DEPLOYMENT_2}, Instance 1)")
    log_result(
        f"1c. Cross-Model Chaining — {DEPLOYMENT} → {DEPLOYMENT_2} (Instance 1)",
        success=True,
        details=f"Chained output: {chained_cross_model.output_text}",
    )
except Exception as e:
    log_result(
        f"1c. Cross-Model Chaining — {DEPLOYMENT} → {DEPLOYMENT_2} (Instance 1)",
        success=False,
        details=str(e),
    )

Retrieving response resp_0da64bbc724270c50069cbbb69401481969b882f95011a7888 from Instance 1 using gpt-5-mini...
  (Response was created with model=gpt-4-1-mini)

📋 Cross-Model Retrieved (gpt-4-1-mini → gpt-5-mini, Instance 1) (JSON):
{
  "id": "resp_0da64bbc724270c50069cbbb69401481969b882f95011a7888",
  "created_at": 1774959465.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-4-1-mini",
  "object": "response",
  "output": [
    {
      "id": "msg_0da64bbc724270c50069cbbb69a3ac81968653570569fe78bf",
      "content": [
        {
          "annotations": [],
          "text": "Paris",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": null
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 1.0,
  "background": false,
  "completed_at": 1774959465.0,

---
## 2. Cross-Instance Retrieve — Retrieve Response from Instance 2 & 3

In [49]:
# Test 2a: Try to retrieve the response created on Instance 1 from Instance 2
print(f"Attempting to retrieve response {response1_id} from Instance 2...")
try:
    retrieved_2 = client2.responses.retrieve(response1_id)
    print_response_json(retrieved_2, "Retrieved Response (Instance 2)")
    log_result(
        "2a. Cross-Instance Retrieve — Instance 1 → Instance 2",
        success=True,
        details=f"Retrieved output: {retrieved_2.output_text}",
    )
except Exception as e:
    log_result(
        "2a. Cross-Instance Retrieve — Instance 1 → Instance 2",
        success=False,
        details=str(e),
    )

Attempting to retrieve response resp_0da64bbc724270c50069cbbb69401481969b882f95011a7888 from Instance 2...
[FAIL] 2a. Cross-Instance Retrieve — Instance 1 → Instance 2
       ↳ Error code: 404 - {'error': {'message': "Response with id 'resp_0da64bbc724270c50069cbbb69401481969b882f95011a7888' not found.", 'type': 'invalid_request_error', 'param': None, 'code': None}}


In [50]:
# Test 2b: Try to retrieve the response created on Instance 1 from Instance 3
print(f"Attempting to retrieve response {response1_id} from Instance 3...")
try:
    retrieved_3 = client3.responses.retrieve(response1_id)
    print_response_json(retrieved_3, "Retrieved Response (Instance 3)")
    log_result(
        "2b. Cross-Instance Retrieve — Instance 1 → Instance 3",
        success=True,
        details=f"Retrieved output: {retrieved_3.output_text}",
    )
except Exception as e:
    log_result(
        "2b. Cross-Instance Retrieve — Instance 1 → Instance 3",
        success=False,
        details=str(e),
    )

Attempting to retrieve response resp_0da64bbc724270c50069cbbb69401481969b882f95011a7888 from Instance 3...
[FAIL] 2b. Cross-Instance Retrieve — Instance 1 → Instance 3
       ↳ Error code: 404 - {'error': {'message': "Response with id 'resp_0da64bbc724270c50069cbbb69401481969b882f95011a7888' not found.", 'type': 'invalid_request_error', 'param': None, 'code': None}}


---
## 3. Cross-Instance Chaining — Use `previous_response_id` on Instance 2 & 3

In [51]:
# Test 3a: Chain on Instance 2 using response_id from Instance 1
print(f"Chaining on Instance 2 with previous_response_id={response1_id}...")
try:
    chained_2 = client2.responses.create(
        model=DEPLOYMENT,
        previous_response_id=response1_id,
        input=[{"role": "user", "content": "What is the population of that city? Answer briefly."}],
    )
    print_response_json(chained_2, "Chained Response (Instance 2)")
    log_result(
        "3a. Cross-Instance Chaining — Instance 1 → Instance 2",
        success=True,
        details=f"Chained output: {chained_2.output_text}",
    )
except Exception as e:
    log_result(
        "3a. Cross-Instance Chaining — Instance 1 → Instance 2",
        success=False,
        details=str(e),
    )

Chaining on Instance 2 with previous_response_id=resp_0da64bbc724270c50069cbbb69401481969b882f95011a7888...
[FAIL] 3a. Cross-Instance Chaining — Instance 1 → Instance 2
       ↳ Error code: 400 - {'error': {'message': "Previous response with id 'resp_0da64bbc724270c50069cbbb69401481969b882f95011a7888' not found.", 'type': 'invalid_request_error', 'param': 'previous_response_id', 'code': 'previous_response_not_found'}}


In [52]:
# Test 3b: Chain on Instance 3 using response_id from Instance 1
print(f"Chaining on Instance 3 with previous_response_id={response1_id}...")
try:
    chained_3 = client3.responses.create(
        model=DEPLOYMENT,
        previous_response_id=response1_id,
        input=[{"role": "user", "content": "What is the population of that city? Answer briefly."}],
    )
    print_response_json(chained_3, "Chained Response (Instance 3)")
    log_result(
        "3b. Cross-Instance Chaining — Instance 1 → Instance 3",
        success=True,
        details=f"Chained output: {chained_3.output_text}",
    )
except Exception as e:
    log_result(
        "3b. Cross-Instance Chaining — Instance 1 → Instance 3",
        success=False,
        details=str(e),
    )

Chaining on Instance 3 with previous_response_id=resp_0da64bbc724270c50069cbbb69401481969b882f95011a7888...
[FAIL] 3b. Cross-Instance Chaining — Instance 1 → Instance 3
       ↳ Error code: 400 - {'error': {'message': "Previous response with id 'resp_0da64bbc724270c50069cbbb69401481969b882f95011a7888' not found.", 'type': 'invalid_request_error', 'param': 'previous_response_id', 'code': 'previous_response_not_found'}}


---
## 4. Background Mode — Baseline (same instance)

In [53]:
# Test 4: Create a background response on Instance 1, poll until complete, retrieve from same instance
print("Creating background response on Instance 1...")
bg_response = client1.responses.create(
    model=DEPLOYMENT,
    input="Explain quantum entanglement in three sentences.",
    background=True,
)

bg_response_id = bg_response.id
print(f"Background Response ID : {bg_response_id}")
print(f"Initial Status         : {bg_response.status}")
print_response_json(bg_response, "Background Create Response (Instance 1)")

# Poll until completion
print("Polling for completion on Instance 1...")
try:
    completed_bg = wait_for_background(client1, bg_response_id)
    print_response_json(completed_bg, "Background Completed Response (Instance 1)")
    log_result(
        "4. Background Baseline — create & poll on Instance 1",
        success=True,
        details=f"Final status: {completed_bg.status} | Output: {completed_bg.output_text[:200]}",
    )
except Exception as e:
    log_result("4. Background Baseline — create & poll on Instance 1", success=False, details=str(e))

Creating background response on Instance 1...
Background Response ID : resp_05bcd7b1b54ce7610069cbbb81474c8194bfc56579e5a43e73
Initial Status         : queued

📋 Background Create Response (Instance 1) (JSON):
{
  "id": "resp_05bcd7b1b54ce7610069cbbb81474c8194bfc56579e5a43e73",
  "created_at": 1774959489.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-4-1-mini",
  "object": "response",
  "output": [],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 1.0,
  "background": true,
  "completed_at": null,
  "conversation": null,
  "max_output_tokens": null,
  "max_tool_calls": null,
  "previous_response_id": null,
  "prompt": null,
  "prompt_cache_key": null,
  "prompt_cache_retention": null,
  "reasoning": {
    "effort": null,
    "generate_summary": null,
    "summary": null
  },
  "safety_identifier": null,
  "service_tier": "auto",
  "status": "queued",
  "text": {
    "for

In [54]:
# Test 4b: Explicitly retrieve the completed background response from Instance 1 (same instance)
print(f"Retrieving background response {bg_response_id} from Instance 1 (same instance)...")
try:
    bg_retrieved_1 = client1.responses.retrieve(bg_response_id)
    print_response_json(bg_retrieved_1, "Background Retrieved (Instance 1 — same instance)")
    log_result(
        "4b. Background Retrieve — same Instance 1",
        success=True,
        details=f"Status: {bg_retrieved_1.status} | Output: {bg_retrieved_1.output_text[:200]}",
    )
except Exception as e:
    log_result(
        "4b. Background Retrieve — same Instance 1",
        success=False,
        details=str(e),
    )

Retrieving background response resp_05bcd7b1b54ce7610069cbbb81474c8194bfc56579e5a43e73 from Instance 1 (same instance)...

📋 Background Retrieved (Instance 1 — same instance) (JSON):
{
  "id": "resp_05bcd7b1b54ce7610069cbbb81474c8194bfc56579e5a43e73",
  "created_at": 1774959489.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-4-1-mini",
  "object": "response",
  "output": [
    {
      "id": "msg_05bcd7b1b54ce7610069cbbb820c448194ad6aaf76cdf92bfe",
      "content": [
        {
          "annotations": [],
          "text": "Quantum entanglement is a phenomenon where two or more particles become interconnected such that the state of one instantly influences the state of the other, no matter how far apart they are. This spooky connection means that measuring one particle's properties immediately determines the properties of its entangled partner. Entanglement challenges classical intuitions about locality and has important applic

In [55]:
# Test 4c: Retrieve the background response (created with gpt-4.1-mini) using gpt-5-mini on Instance 1
print(f"Retrieving background response {bg_response_id} from Instance 1 using {DEPLOYMENT_2}...")
print(f"  (Background response was created with model={DEPLOYMENT})")
try:
    bg_cross_model = client1.responses.retrieve(bg_response_id)
    print_response_json(bg_cross_model, f"Background Cross-Model Retrieved ({DEPLOYMENT} → {DEPLOYMENT_2}, Instance 1)")
    log_result(
        f"4c. Background Cross-Model Retrieve — {DEPLOYMENT} → {DEPLOYMENT_2} (Instance 1)",
        success=True,
        details=f"Status: {bg_cross_model.status} | Output: {bg_cross_model.output_text[:200]}",
    )
except Exception as e:
    log_result(
        f"4c. Background Cross-Model Retrieve — {DEPLOYMENT} → {DEPLOYMENT_2} (Instance 1)",
        success=False,
        details=str(e),
    )

# Test 4d: Chain on Instance 1 using gpt-5-mini with background previous_response_id from gpt-4.1-mini
print(f"\nChaining on Instance 1 with {DEPLOYMENT_2} using background previous_response_id={bg_response_id}...")
print(f"  (Background response was created with model={DEPLOYMENT})")
try:
    bg_chained_cross_model = client1.responses.create(
        model=DEPLOYMENT_2,
        previous_response_id=bg_response_id,
        input=[{"role": "user", "content": "Can you give a real-world analogy for that concept?"}],
    )
    print_response_json(bg_chained_cross_model, f"Background Cross-Model Chained ({DEPLOYMENT} → {DEPLOYMENT_2}, Instance 1)")
    log_result(
        f"4d. Background Cross-Model Chaining — {DEPLOYMENT} → {DEPLOYMENT_2} (Instance 1)",
        success=True,
        details=f"Chained output: {bg_chained_cross_model.output_text[:200]}",
    )
except Exception as e:
    log_result(
        f"4d. Background Cross-Model Chaining — {DEPLOYMENT} → {DEPLOYMENT_2} (Instance 1)",
        success=False,
        details=str(e),
    )

Retrieving background response resp_05bcd7b1b54ce7610069cbbb81474c8194bfc56579e5a43e73 from Instance 1 using gpt-5-mini...
  (Background response was created with model=gpt-4-1-mini)

📋 Background Cross-Model Retrieved (gpt-4-1-mini → gpt-5-mini, Instance 1) (JSON):
{
  "id": "resp_05bcd7b1b54ce7610069cbbb81474c8194bfc56579e5a43e73",
  "created_at": 1774959489.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-4-1-mini",
  "object": "response",
  "output": [
    {
      "id": "msg_05bcd7b1b54ce7610069cbbb820c448194ad6aaf76cdf92bfe",
      "content": [
        {
          "annotations": [],
          "text": "Quantum entanglement is a phenomenon where two or more particles become interconnected such that the state of one instantly influences the state of the other, no matter how far apart they are. This spooky connection means that measuring one particle's properties immediately determines the properties of its entangled partner. 

---
## 5. Background Cross-Instance Retrieve

In [56]:
# Test 5a: Retrieve the background response (created on Instance 1) from Instance 2
print(f"Attempting to retrieve background response {bg_response_id} from Instance 2...")
try:
    bg_retrieved_2 = client2.responses.retrieve(bg_response_id)
    print_response_json(bg_retrieved_2, "Background Retrieved (Instance 2)")
    log_result(
        "5a. Background Cross-Instance Retrieve — Instance 1 → Instance 2",
        success=True,
        details=f"Status: {bg_retrieved_2.status} | Output: {bg_retrieved_2.output_text[:200]}",
    )
except Exception as e:
    log_result(
        "5a. Background Cross-Instance Retrieve — Instance 1 → Instance 2",
        success=False,
        details=str(e),
    )

Attempting to retrieve background response resp_05bcd7b1b54ce7610069cbbb81474c8194bfc56579e5a43e73 from Instance 2...
[FAIL] 5a. Background Cross-Instance Retrieve — Instance 1 → Instance 2
       ↳ Error code: 404 - {'error': {'message': "Response with id 'resp_05bcd7b1b54ce7610069cbbb81474c8194bfc56579e5a43e73' not found.", 'type': 'invalid_request_error', 'param': None, 'code': None}}


In [57]:
# Test 5b: Retrieve the background response (created on Instance 1) from Instance 3
print(f"Attempting to retrieve background response {bg_response_id} from Instance 3...")
try:
    bg_retrieved_3 = client3.responses.retrieve(bg_response_id)
    print_response_json(bg_retrieved_3, "Background Retrieved (Instance 3)")
    log_result(
        "5b. Background Cross-Instance Retrieve — Instance 1 → Instance 3",
        success=True,
        details=f"Status: {bg_retrieved_3.status} | Output: {bg_retrieved_3.output_text[:200]}",
    )
except Exception as e:
    log_result(
        "5b. Background Cross-Instance Retrieve — Instance 1 → Instance 3",
        success=False,
        details=str(e),
    )

Attempting to retrieve background response resp_05bcd7b1b54ce7610069cbbb81474c8194bfc56579e5a43e73 from Instance 3...
[FAIL] 5b. Background Cross-Instance Retrieve — Instance 1 → Instance 3
       ↳ Error code: 404 - {'error': {'message': "Response with id 'resp_05bcd7b1b54ce7610069cbbb81474c8194bfc56579e5a43e73' not found.", 'type': 'invalid_request_error', 'param': None, 'code': None}}


---
## 6. Background Cross-Instance Chaining

In [58]:
# Test 6a: Chain on Instance 2 using the background response_id from Instance 1
print(f"Chaining on Instance 2 with background previous_response_id={bg_response_id}...")
try:
    bg_chained_2 = client2.responses.create(
        model=DEPLOYMENT,
        previous_response_id=bg_response_id,
        input=[{"role": "user", "content": "Can you give a real-world analogy for that concept?"}],
    )
    print_response_json(bg_chained_2, "Background Chained Response (Instance 2)")
    log_result(
        "6a. Background Cross-Instance Chaining — Instance 1 → Instance 2",
        success=True,
        details=f"Chained output: {bg_chained_2.output_text[:200]}",
    )
except Exception as e:
    log_result(
        "6a. Background Cross-Instance Chaining — Instance 1 → Instance 2",
        success=False,
        details=str(e),
    )

Chaining on Instance 2 with background previous_response_id=resp_05bcd7b1b54ce7610069cbbb81474c8194bfc56579e5a43e73...
[FAIL] 6a. Background Cross-Instance Chaining — Instance 1 → Instance 2
       ↳ Error code: 400 - {'error': {'message': "Previous response with id 'resp_05bcd7b1b54ce7610069cbbb81474c8194bfc56579e5a43e73' not found.", 'type': 'invalid_request_error', 'param': 'previous_response_id', 'code': 'previous_response_not_found'}}


In [59]:
# Test 6b: Chain on Instance 3 using the background response_id from Instance 1
print(f"Chaining on Instance 3 with background previous_response_id={bg_response_id}...")
try:
    bg_chained_3 = client3.responses.create(
        model=DEPLOYMENT,
        previous_response_id=bg_response_id,
        input=[{"role": "user", "content": "Can you give a real-world analogy for that concept?"}],
    )
    print_response_json(bg_chained_3, "Background Chained Response (Instance 3)")
    log_result(
        "6b. Background Cross-Instance Chaining — Instance 1 → Instance 3",
        success=True,
        details=f"Chained output: {bg_chained_3.output_text[:200]}",
    )
except Exception as e:
    log_result(
        "6b. Background Cross-Instance Chaining — Instance 1 → Instance 3",
        success=False,
        details=str(e),
    )

Chaining on Instance 3 with background previous_response_id=resp_05bcd7b1b54ce7610069cbbb81474c8194bfc56579e5a43e73...
[FAIL] 6b. Background Cross-Instance Chaining — Instance 1 → Instance 3
       ↳ Error code: 400 - {'error': {'message': "Previous response with id 'resp_05bcd7b1b54ce7610069cbbb81474c8194bfc56579e5a43e73' not found.", 'type': 'invalid_request_error', 'param': 'previous_response_id', 'code': 'previous_response_not_found'}}


---
## 7. Summary of Results

In [60]:
print("=" * 100)
print("RESPONSE API STATELESS CROSS-INSTANCE & CROSS-MODEL TEST RESULTS")
print("=" * 100)
print(f"{'Test':<68} {'Status':<8} Details")
print("-" * 100)
for r in results:
    colour = "\033[92m" if r['status'] == 'PASS' else "\033[91m"
    # Truncate details for the summary table
    short_details = (r['details'][:55] + '...') if len(r['details']) > 55 else r['details']
    print(f"{r['test']:<68} {colour}{r['status']:<8}\033[0m {short_details}")
print("-" * 100)

passed = sum(1 for r in results if r['status'] == 'PASS')
failed = sum(1 for r in results if r['status'] == 'FAIL')
print(f"\nTotal: {len(results)} | Passed: {passed} | Failed: {failed}")

if failed > 0:
    print("\n⚠️  Some tests FAILED — response state is NOT shared across instances/models.")
    print("   This confirms that each Azure OpenAI resource maintains its own response storage.")
else:
    print("\n✅ All tests PASSED — response state IS shared across instances and models.")

RESPONSE API STATELESS CROSS-INSTANCE & CROSS-MODEL TEST RESULTS
Test                                                                 Status   Details
----------------------------------------------------------------------------------------------------
1. Baseline — create & retrieve on Instance 1                        PASS     Retrieved output: Paris
1b. Cross-Model Retrieve — gpt-4-1-mini → gpt-5-mini (Instance 1)    PASS     Retrieved output: Paris
1c. Cross-Model Chaining — gpt-4-1-mini → gpt-5-mini (Instance 1)    PASS     Chained output: About 2.1 million (city proper).
2a. Cross-Instance Retrieve — Instance 1 → Instance 2                FAIL     Error code: 404 - {'error': {'message': "Response with ...
2b. Cross-Instance Retrieve — Instance 1 → Instance 3                FAIL     Error code: 404 - {'error': {'message': "Response with ...
3a. Cross-Instance Chaining — Instance 1 → Instance 2                FAIL     Error code: 400 - {'error': {'message': "Previous respo...
3b. Cr

---
## 8. Cleanup (Optional)

Delete responses created during testing to keep storage clean.

In [61]:
# Cleanup: Delete responses from Instance 1
print("Cleaning up responses on Instance 1...")
try:
    # client1.responses.delete(response1_id)
    print(f"  Deleted {response1_id}")
except Exception as e:
    print(f"  Could not delete {response1_id}: {e}")

try:
    # client1.responses.delete(bg_response_id)
    print(f"  Deleted {bg_response_id}")
except Exception as e:
    print(f"  Could not delete {bg_response_id}: {e}")

print("Done.")

Cleaning up responses on Instance 1...
  Deleted resp_0da64bbc724270c50069cbbb69401481969b882f95011a7888
  Deleted resp_05bcd7b1b54ce7610069cbbb81474c8194bfc56579e5a43e73
Done.
